In [1]:
!rm /kaggle/working/cgl_cache/ceemdan_*.npy
!rm /kaggle/working/cgl_cache/lstm_*.keras
!rm /kaggle/working/cgl_cache/sc_*.pkl
!rm /kaggle/working/cgl_cache/ens_*.keras
!rm /kaggle/working/cgl_cache/hp_imf_*.pkl
!rm /kaggle/working/cgl_checkpoints/*.pkl
# keep prices_*.pkl

rm: cannot remove '/kaggle/working/cgl_cache/ceemdan_*.npy': No such file or directory
rm: cannot remove '/kaggle/working/cgl_cache/lstm_*.keras': No such file or directory
rm: cannot remove '/kaggle/working/cgl_cache/sc_*.pkl': No such file or directory
rm: cannot remove '/kaggle/working/cgl_cache/ens_*.keras': No such file or directory
rm: cannot remove '/kaggle/working/cgl_cache/hp_imf_*.pkl': No such file or directory
rm: cannot remove '/kaggle/working/cgl_checkpoints/*.pkl': No such file or directory


In [2]:
# =============================================================================
# CGL-BL REPLICATION v2 — ALL FIXES + DUAL GPU PARALLELIZATION
#
# FIXES vs previous run:
#   FIX A — Full-series CEEMDAN  : decompose train+test together (paper-faithful)
#   FIX B — Proper rolling window : test step t uses IMF[t-LOOKBACK:t] not static end
#   FIX C — Wider HP ranges       : batch [2,64], neurons [4,256] (paper-exact)
#   FIX D — Dual GPU              : GPU:0 → SSE stocks | GPU:1 → DJIA stocks
#                                   via multiprocessing.Process — true parallelism
#
# RUNTIME ESTIMATE (2× Tesla T4):
#   SSE (P1+P2) on GPU:0 : ~7 hrs   |   DJIA on GPU:1 : ~3.5 hrs  (parallel)
#   Total wall-clock      : ~7 hrs first run | ~2 hrs on re-run (all cached)
#
# IMPORTANT: Clear all cgl_cache files before running (CEEMDAN/GLSTM/HP/Ensemble)
#            Price download cache can be kept.
# =============================================================================

# ── CELL 0: Install ───────────────────────────────────────────────────────────
!pip install EMD-signal yfinance optuna tqdm scikit-learn -q

# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import os, sys, time, warnings, random, pickle, hashlib
import multiprocessing as mp
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.optimize import minimize

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import yfinance as yf
from PyEMD import CEEMDAN as CEEMDAN_lib
from tqdm.auto import tqdm

# =============================================================================
# CONSTANTS — PAPER EXACT
# =============================================================================
LAMBDA      = 2.5
TAU         = 0.025
LOOKBACK    = 10
TRANS_COST  = 0.002
EPOCHS      = 100
PATIENCE    = 10

# FIX C — widened to match paper (Section 4.1.1 Stage 2)
BATCH_SIZES  = [2, 4, 8, 16, 32, 64]   # paper: [2, 64]
N_UNITS_MIN  = 4                         # paper: [4, 256]
N_UNITS_MAX  = 256
N_TRIALS_IMF = 5                         # increased from 3 — still fast per IMF

CACHE_DIR = '/kaggle/working/cgl_cache'
CKPT_DIR  = '/kaggle/working/cgl_checkpoints'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

SEED = 42
np.random.seed(SEED); random.seed(SEED)

print(f"Config: LOOKBACK={LOOKBACK} | N_TRIALS_IMF={N_TRIALS_IMF} | "
      f"λ={LAMBDA} | τ={TAU} | TC={TRANS_COST*100:.1f}%")
print(f"Batch range: {BATCH_SIZES} | Units: [{N_UNITS_MIN}, {N_UNITS_MAX}]")
print(f"Cache : {CACHE_DIR}")
print(f"Ckpts : {CKPT_DIR}")


# =============================================================================
# STOCKS & WEIGHTS (Fig. 6)
# =============================================================================
SSE_TICKERS = {
    '600519':'600519.SS','601318':'601318.SS','600036':'600036.SS',
    '601166':'601166.SS','600900':'600900.SS','600276':'600276.SS',
    '600030':'600030.SS','601012':'601012.SS','600887':'600887.SS',
    '601398':'601398.SS',
}
SSE_MKT_W_RAW = {
    '600519':0.5017,'601318':0.0833,'600036':0.0437,'601166':0.0431,
    '600900':0.0413,'600276':0.0401,'600030':0.0388,'601012':0.0366,
    '600887':0.0333,'601398':0.0266,
}
DJIA_TICKERS = {
    'UNH':'UNH','MSFT':'MSFT','HD':'HD','V':'V','JPM':'JPM',
    'AAPL':'AAPL','AMZN':'AMZN','PG':'PG','JNJ':'JNJ','WMT':'WMT',
}
DJIA_MKT_W_RAW = {
    'UNH':0.0768,'MSFT':0.0527,'HD':0.0508,'V':0.0361,'JPM':0.0325,
    'AAPL':0.0293,'AMZN':0.0245,'PG':0.0237,'JNJ':0.0191,'WMT':0.0121,
}
def norm_w(d): t=sum(d.values()); return {k:v/t for k,v in d.items()}
SSE_MKT_W  = norm_w(SSE_MKT_W_RAW)
DJIA_MKT_W = norm_w(DJIA_MKT_W_RAW)

SSE_PERIODS = {
    'P1':{'ts':'2022-01-04','te':'2023-12-29',
          'vs':'2024-01-02','ve':'2024-03-29','label':'Period 1 (Uptrend)'},
    'P2':{'ts':'2022-01-04','te':'2024-03-29',
          'vs':'2024-04-01','ve':'2024-06-28','label':'Period 2 (Volatile)'},
    # Uncomment after P1+P2 confirmed (~+3.5 hrs wall clock)
    # 'P3':{'ts':'2022-01-04','te':'2024-06-28',
    #       'vs':'2024-07-01','ve':'2024-09-20','label':'Period 3 (Downtrend)'},
}
DJIA_CFG = {
    'ts':'2019-01-01','te':'2023-12-31',
    'vs':'2024-01-01','ve':'2024-12-31'
}


# =============================================================================
# DATA DOWNLOAD
# =============================================================================
def download_data(tickers_dict, start, end, interval='1d'):
    ck = hashlib.md5(
        f"{sorted(tickers_dict.items())}{start}{end}{interval}".encode()
    ).hexdigest()[:10]
    cf = f"{CACHE_DIR}/prices_{ck}.pkl"
    if os.path.exists(cf):
        print("  → from cache")
        with open(cf,'rb') as f: return pickle.load(f)

    all_sym  = list(tickers_dict.values())
    name_map = {v:k for k,v in tickers_dict.items()}
    prices   = {}
    raw = yf.download(all_sym, start=start, end=end, interval=interval,
                      auto_adjust=True, progress=False, group_by='ticker')

    for sym in all_sym:
        short = name_map[sym]
        try:
            if isinstance(raw.columns, pd.MultiIndex):
                s = raw[(sym,'Close')].dropna() \
                    if (sym,'Close') in raw.columns \
                    else raw.xs('Close',axis=1,level=1)[sym].dropna()
            else:
                s = raw['Close'].dropna()
            if len(s)>20: prices[short]=s
        except:
            try:
                df = yf.download(sym,start=start,end=end,interval=interval,
                                  auto_adjust=True,progress=False)
                s = (df.xs('Close',axis=1,level=1).iloc[:,0]
                     if isinstance(df.columns,pd.MultiIndex)
                     else df['Close']).dropna()
                if len(s)>20: prices[short]=s
            except: pass

    prices_df  = pd.DataFrame(prices).dropna(how='all').ffill().bfill()
    returns_df = prices_df.pct_change().dropna()
    result = (prices_df, returns_df)
    with open(cf,'wb') as f: pickle.dump(result,f)
    print(f"  ✓ {len(prices)} tickers | returns: {returns_df.shape}")
    return result

def split_ret(ret, ts, te, vs, ve):
    return ret.loc[ts:te].copy(), ret.loc[vs:ve].copy()


# =============================================================================
# FIX A — FULL-SERIES CEEMDAN
# Decomposes train+test together so IMFs for test period are pre-computed.
# This matches the paper's approach (Fig. 3 shows ~600+ points = full series).
# =============================================================================
def ceemdan_full_series(full_arr, stock_name, period_key):
    """
    FIX A: CEEMDAN on the FULL series (train + test concatenated).
    Returns all IMFs + residual, shape (n_comps, T_full).
    Cached by hash of full array.
    """
    h  = hashlib.md5(full_arr.tobytes()).hexdigest()[:10]
    cf = f"{CACHE_DIR}/ceemdan_full_{stock_name}_{period_key}_{h}.npy"
    if os.path.exists(cf):
        return np.load(cf)
    cem   = CEEMDAN_lib(trials=100, epsilon=0.005)
    imfs  = cem.ceemdan(full_arr.astype(np.float64))
    resid = full_arr - imfs.sum(axis=0)
    comps = np.vstack([imfs, resid.reshape(1,-1)])
    np.save(cf, comps)
    return comps


# =============================================================================
# SEQUENCES & LSTM BUILDER
# =============================================================================
def make_sequences(arr, lookback=LOOKBACK, scaler=None, fit=True):
    arr = np.array(arr, dtype=np.float32)
    if scaler is None: scaler = StandardScaler()
    if fit: scaler.fit(arr.reshape(-1,1))
    s    = scaler.transform(arr.reshape(-1,1)).flatten()
    X, y = [], []
    for i in range(lookback, len(s)):
        X.append(s[i-lookback:i]); y.append(s[i])
    return (np.array(X, dtype=np.float32)[..., np.newaxis],
            np.array(y, dtype=np.float32), scaler)

def build_lstm(n_layers, n_units, lr, gpu_id=0,
               lookback=LOOKBACK, n_features=1):
    """Build LSTM pinned to a specific GPU."""
    with tf.device(f'/GPU:{gpu_id}'):
        inp = Input(shape=(lookback, n_features))
        x   = inp
        for i in range(n_layers):
            x = LSTM(n_units, return_sequences=(i < n_layers-1))(x)
            x = Dropout(0.1)(x)
        out = Dense(1)(x)
        m   = Model(inp, out)
        m.compile(optimizer=Adam(lr), loss='mse')
    return m

def train_model(model, X, y, batch_size, epochs=EPOCHS, patience=PATIENCE):
    n_val = max(int(len(X)*0.15), 2)
    cbs   = [
        EarlyStopping(monitor='val_loss', patience=patience,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=0),
    ]
    model.fit(X[:-n_val], y[:-n_val],
              validation_data=(X[-n_val:], y[-n_val:]),
              epochs=epochs, batch_size=max(batch_size, 2),
              callbacks=cbs, verbose=0)
    return model


# =============================================================================
# FIX C — PER-IMF HP SEARCH WITH PAPER-EXACT RANGES
# =============================================================================
def get_best_hp_for_imf(imf_arr, imf_idx, stock_name, period_key,
                         gpu_id=0, n_trials=N_TRIALS_IMF):
    """
    Independent Optuna HP search per IMF.
    FIX C: batch [2,64], units [4,256] — paper-exact ranges.
    """
    hp_cf = (f"{CACHE_DIR}/hp_imf_{stock_name}_{period_key}"
             f"_imf{imf_idx}.pkl")
    if os.path.exists(hp_cf):
        with open(hp_cf,'rb') as f: return pickle.load(f)

    X, y, _ = make_sequences(imf_arr)
    n_val    = max(int(len(X)*0.2), 2)
    Xtr, Xv  = X[:-n_val], X[-n_val:]
    ytr, yv  = y[:-n_val], y[-n_val:]

    def objective(trial):
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED + imf_idx)
        lr  = trial.suggest_float('lr', 0.001, 0.01, log=True)
        nl  = trial.suggest_int('n_layers', 1, 3)
        nu  = trial.suggest_int('n_units', N_UNITS_MIN, N_UNITS_MAX)
        bs  = trial.suggest_categorical('batch_size', BATCH_SIZES)
        try:
            m  = build_lstm(nl, nu, lr, gpu_id=gpu_id)
            es = EarlyStopping(monitor='val_loss', patience=5,
                               restore_best_weights=True)
            m.fit(Xtr, ytr, epochs=50, batch_size=bs,
                  validation_data=(Xv, yv), callbacks=[es], verbose=0)
            return -float(mean_squared_error(
                yv, m.predict(Xv, verbose=0).flatten()))
        except: return -999.0
        finally: tf.keras.backend.clear_session()

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED + imf_idx),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    with open(hp_cf,'wb') as f: pickle.dump(best, f)
    return best


# =============================================================================
# TRAIN GLSTM ON TRAINING-PORTION OF FULL-SERIES IMF
# =============================================================================
def train_glstm_for_imf(imf_train_arr, hp, stock_name, period_key,
                         imf_idx, gpu_id=0):
    """
    Train GLSTM on the training portion of the full-series IMF.
    Cached to disk.
    """
    mp_path = (f"{CACHE_DIR}/lstm_{stock_name}_{period_key}"
               f"_imf{imf_idx}.keras")
    sp_path = (f"{CACHE_DIR}/sc_{stock_name}_{period_key}"
               f"_imf{imf_idx}.pkl")

    if os.path.exists(mp_path) and os.path.exists(sp_path):
        m = tf.keras.models.load_model(mp_path)
        with open(sp_path,'rb') as f: sc = pickle.load(f)
        return m, sc

    X, y, sc = make_sequences(imf_train_arr)
    m = build_lstm(hp['n_layers'], hp['n_units'], hp['lr'], gpu_id=gpu_id)
    m = train_model(m, X, y, hp['batch_size'])
    m.save(mp_path)
    with open(sp_path,'wb') as f: pickle.dump(sc, f)
    return m, sc


# =============================================================================
# ENSEMBLE LSTM
# =============================================================================
def build_ensemble_input(ens_mat, train_arr, lookback=LOOKBACK):
    """Build (X_ens, y_ens) from IMF prediction matrix."""
    T, n_imfs = ens_mat.shape
    X_ens, y_ens = [], []
    for t in range(lookback, T):
        X_ens.append(ens_mat[t-lookback:t, :])
        y_ens.append(train_arr[lookback + t])
    return (np.array(X_ens, dtype=np.float32),
            np.array(y_ens, dtype=np.float32))

def train_ensemble_lstm(X_ens, y_ens, n_imfs,
                         stock_name, period_key, gpu_id=0):
    """Train Ensemble LSTM. Cached."""
    ens_cf = f"{CACHE_DIR}/ens_{stock_name}_{period_key}.keras"
    scX_cf = f"{CACHE_DIR}/ens_scX_{stock_name}_{period_key}.pkl"
    scY_cf = f"{CACHE_DIR}/ens_scY_{stock_name}_{period_key}.pkl"

    if os.path.exists(ens_cf) and os.path.exists(scX_cf):
        ens_m = tf.keras.models.load_model(ens_cf)
        with open(scX_cf,'rb') as f: sc_X = pickle.load(f)
        with open(scY_cf,'rb') as f: sc_y = pickle.load(f)
        print(f"  ║    Ensemble LSTM → from cache")
        return ens_m, sc_X, sc_y

    tf.keras.backend.clear_session()
    sc_X = StandardScaler()
    sc_y = StandardScaler()
    N, L, F = X_ens.shape
    X_s = sc_X.fit_transform(
        X_ens.reshape(-1,F)).reshape(N,L,F).astype(np.float32)
    y_s = sc_y.fit_transform(
        y_ens.reshape(-1,1)).flatten().astype(np.float32)

    with tf.device(f'/GPU:{gpu_id}'):
        inp   = Input(shape=(L, F))
        x     = LSTM(32, return_sequences=False)(inp)
        x     = Dropout(0.1)(x)
        out   = Dense(1)(x)
        ens_m = Model(inp, out)
        ens_m.compile(optimizer=Adam(1e-3), loss='mse')

    n_val = max(int(len(X_s)*0.15), 2)
    cbs   = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=0),
    ]
    ens_m.fit(X_s[:-n_val], y_s[:-n_val],
              validation_data=(X_s[-n_val:], y_s[-n_val:]),
              epochs=EPOCHS, batch_size=32, callbacks=cbs, verbose=0)

    # Val R²
    p_s  = ens_m.predict(X_s[-n_val:], verbose=0).flatten()
    pred = sc_y.inverse_transform(p_s.reshape(-1,1)).flatten()
    true = sc_y.inverse_transform(y_s[-n_val:].reshape(-1,1)).flatten()
    print(f"  ║    Ensemble LSTM val R²={r2_score(true,pred):.4f}")

    ens_m.save(ens_cf)
    with open(scX_cf,'wb') as f: pickle.dump(sc_X, f)
    with open(scY_cf,'wb') as f: pickle.dump(sc_y, f)
    return ens_m, sc_X, sc_y


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================
def ckpt_path(stock_name, period_key):
    return f"{CKPT_DIR}/pred_{stock_name}_{period_key}.pkl"

def save_checkpoint(stock_name, period_key, preds, actuals, metrics):
    with open(ckpt_path(stock_name, period_key),'wb') as f:
        pickle.dump({'preds':preds,'actuals':actuals,'metrics':metrics},f)

def load_checkpoint(stock_name, period_key):
    cp = ckpt_path(stock_name, period_key)
    if os.path.exists(cp):
        with open(cp,'rb') as f: d = pickle.load(f)
        return d['preds'], d['actuals'], d['metrics']
    return None


# =============================================================================
# FULL CGL PIPELINE — ALL FIXES APPLIED
# =============================================================================
def cgl_predict(train_series, test_series, stock_name, period_key,
                gpu_id=0):
    """
    Paper-faithful CGL pipeline with all fixes:

    FIX A — Full-series CEEMDAN: decompose train+test together
    FIX B — Proper rolling window: test step t uses IMF[T_train+t-L : T_train+t]
    FIX C — Wider HP ranges: batch [2,64], neurons [4,256]
    """
    t0        = time.time()
    train_arr = np.array(train_series, dtype=np.float64)
    test_arr  = np.array(test_series,  dtype=np.float64)
    T_train   = len(train_arr)
    T_test    = len(test_arr)

    # GPU setup inside worker process
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            tf.config.set_visible_devices(gpus[gpu_id], 'GPU')
            tf.config.experimental.set_memory_growth(gpus[gpu_id], True)
        except: pass

    tf.random.set_seed(SEED)

    print(f"\n  ╔══ [GPU:{gpu_id}] [{stock_name}] {period_key} ══")

    # ── FIX A: CEEMDAN on full series (train + test) ──────────────────────────
    print(f"  ║  Stage 1 — CEEMDAN (FULL series: {T_train}+{T_test}={T_train+T_test} pts)",
          end='', flush=True)

    full_arr = np.concatenate([train_arr, test_arr])
    comps    = ceemdan_full_series(full_arr, stock_name, period_key)
    n_comps  = comps.shape[0]
    print(f"  → {n_comps} components")

    # Split IMFs: training portion and test portion
    # Training IMFs: columns 0 : T_train
    # Test IMFs    : columns T_train : T_train + T_test
    imfs_train = comps[:, :T_train]   # (n_comps, T_train)
    imfs_test  = comps[:, T_train:]   # (n_comps, T_test)
    imfs_full  = comps               # (n_comps, T_full) — for rolling window

    # ── Stage 2+3: Per-IMF HP tuning + GLSTM training ────────────────────────
    print(f"  ║  Stage 2+3 — Per-IMF HP tuning + GLSTM (FIX C: batch[2,64] units[4,256])")
    models, scalers = [], []

    imf_pbar = tqdm(range(n_comps),
                    desc=f"  ║    GLSTM [GPU:{gpu_id}] [{stock_name}]",
                    leave=False, unit="IMF")

    for i in imf_pbar:
        imf_pbar.set_postfix_str(f"IMF {i+1}/{n_comps} — HP search")
        hp = get_best_hp_for_imf(
            imfs_train[i], i, stock_name, period_key,
            gpu_id=gpu_id, n_trials=N_TRIALS_IMF
        )
        imf_pbar.set_postfix_str(f"IMF {i+1}/{n_comps} — training")
        m, sc = train_glstm_for_imf(
            imfs_train[i], hp, stock_name, period_key, i, gpu_id=gpu_id
        )
        models.append(m); scalers.append(sc)

        # Quick val R²
        X_i, y_i, _ = make_sequences(imfs_train[i])
        n_v  = max(int(len(X_i)*0.2), 2)
        p_s  = m.predict(X_i[-n_v:], verbose=0).flatten()
        pred = sc.inverse_transform(p_s.reshape(-1,1)).flatten()
        true = sc.inverse_transform(y_i[-n_v:].reshape(-1,1)).flatten()
        imf_pbar.set_postfix_str(
            f"IMF {i+1} hp={hp} R²={r2_score(true,pred):.3f}")

    # ── Stage 4: IMF prediction matrix on TRAINING portion ───────────────────
    print(f"  ║  Stage 4 — Training IMF prediction matrix "
          f"({T_train-LOOKBACK} × {n_comps})")
    ens_mat_train = np.zeros(
        (T_train - LOOKBACK, n_comps), dtype=np.float32)

    for i, (m, sc) in enumerate(zip(models, scalers)):
        sc_s  = sc.transform(imfs_train[i].reshape(-1,1)).flatten()
        X_all = np.array(
            [sc_s[t-LOOKBACK:t] for t in range(LOOKBACK, T_train)],
            dtype=np.float32)[..., np.newaxis]
        p_s   = m.predict(X_all, verbose=0, batch_size=256).flatten()
        ens_mat_train[:,i] = sc.inverse_transform(
            p_s.reshape(-1,1)).flatten()

    # ── Stage 5: Ensemble LSTM ────────────────────────────────────────────────
    print(f"  ║  Stage 5 — Ensemble LSTM [PAPER CORE]")
    X_ens, y_ens = build_ensemble_input(ens_mat_train, train_arr, LOOKBACK)
    ens_m, sc_X, sc_y = train_ensemble_lstm(
        X_ens, y_ens, n_comps, stock_name, period_key, gpu_id=gpu_id)

    # ── FIX B: Stage 6 — Proper rolling window test prediction ───────────────
    # For test step t (0-indexed):
    #   Input window for IMF i = imfs_full[i, T_train+t-LOOKBACK : T_train+t]
    # This uses the pre-computed full-series IMF values at the correct position.
    # No more static end-of-training window — window slides forward each step.
    print(f"  ║  Stage 6 — Test prediction (FIX B: rolling IMF window, "
          f"{T_test} steps)")

    # Initialize ensemble buffer from end of training predictions
    ens_buffer = ens_mat_train[-LOOKBACK:].copy()  # (LOOKBACK, n_comps)

    test_preds = []
    test_pbar  = tqdm(range(T_test),
                      desc=f"  ║    Predict [GPU:{gpu_id}] [{stock_name}]",
                      leave=False, unit="step")

    for step in test_pbar:
        # FIX B: compute input window start/end in full series
        # At step t, the window covers full series positions:
        # [T_train + step - LOOKBACK  :  T_train + step]
        win_start = T_train + step - LOOKBACK
        win_end   = T_train + step

        # Guard: if step < LOOKBACK, pad with training IMF values
        if win_start < 0:
            # Should not happen since T_train >> LOOKBACK, but safety guard
            win_start = 0

        step_imf = np.zeros(n_comps, dtype=np.float32)
        for i, (m, sc) in enumerate(zip(models, scalers)):
            # Extract the correct LOOKBACK-length window from full-series IMF
            window = imfs_full[i, win_start:win_end]

            # If window is shorter than LOOKBACK (edge case), pad left
            if len(window) < LOOKBACK:
                pad    = np.zeros(LOOKBACK - len(window))
                window = np.concatenate([pad, window])

            win_s  = sc.transform(
                window.reshape(-1,1)
            ).reshape(1, LOOKBACK, 1).astype(np.float32)
            p_s    = m.predict(win_s, verbose=0).flatten()[0]
            step_imf[i] = float(sc.inverse_transform([[p_s]])[0,0])

        # Roll ensemble buffer and run Ensemble LSTM
        ens_buffer = np.vstack([ens_buffer[1:], step_imf.reshape(1,-1)])
        buf_s  = sc_X.transform(
            ens_buffer.reshape(-1, n_comps)
        ).reshape(1, LOOKBACK, n_comps).astype(np.float32)
        pred_s = float(ens_m.predict(buf_s, verbose=0).flatten()[0])
        pred   = float(sc_y.inverse_transform([[pred_s]])[0,0])
        test_preds.append(pred)

        test_pbar.set_postfix_str(
            f"pred={pred:.5f} | true={test_arr[step]:.5f}")

    # Metrics
    test_preds = np.array(test_preds, dtype=np.float32)
    test_arr_f = test_arr.astype(np.float32)
    mae  = mean_absolute_error(test_arr_f, test_preds)
    rmse = np.sqrt(mean_squared_error(test_arr_f, test_preds))
    r2   = r2_score(test_arr_f, test_preds)

    print(f"  ╚══ [GPU:{gpu_id}] [{stock_name}] ✓  "
          f"MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  "
          f"({time.time()-t0:.0f}s)")

    return test_preds, test_arr_f, {'MAE':mae,'RMSE':rmse,'R2':r2}


# =============================================================================
# BLACK-LITTERMAN (Eq. 4–6)
# =============================================================================
def black_litterman(ret_df, view_dict, mkt_w, lam=LAMBDA, tau=TAU):
    assets = list(ret_df.columns); N = len(assets)
    views  = [(k,v) for k,v in view_dict.items() if k in assets]
    K      = len(views)
    Sigma  = ret_df.cov().values + np.eye(N)*1e-8
    w_mkt  = np.array([mkt_w.get(a,1/N) for a in assets])
    w_mkt /= w_mkt.sum()
    Pi     = lam * Sigma @ w_mkt
    if K == 0:
        w = np.linalg.inv(lam*Sigma) @ Pi
    else:
        P = np.zeros((K,N)); Q = np.zeros(K)
        for k,(s,r) in enumerate(views):
            P[k,assets.index(s)] = 1.0; Q[k] = r
        Omega = np.diag([tau*float(P[k]@Sigma@P[k])
                         for k in range(K)]) + np.eye(K)*1e-8
        tSi   = np.linalg.inv(tau*Sigma)
        Oi    = np.linalg.inv(Omega)
        mu    = np.linalg.inv(tSi+P.T@Oi@P)@(tSi@Pi+P.T@Oi@Q)
        w     = np.linalg.inv(lam*Sigma) @ mu
    w = np.maximum(w,0); w = w/(w.sum()+1e-9)
    return pd.Series(w, index=assets)

def mean_variance_weights(ret_df):
    mu  = ret_df.mean().values; S = ret_df.cov().values; N = len(mu)
    res = minimize(
        lambda w: -(w@mu)/(np.sqrt(w@S@w)+1e-9),
        np.ones(N)/N, method='SLSQP',
        bounds=[(0,1)]*N,
        constraints={'type':'eq','fun':lambda w:w.sum()-1},
        options={'maxiter':1000,'ftol':1e-9}
    )
    w = res.x if res.success else np.ones(N)/N
    return pd.Series(w/w.sum(), index=ret_df.columns)


# =============================================================================
# PORTFOLIO METRICS (Eq. 22–24)
# =============================================================================
def portfolio_metrics(returns, ann=252):
    r   = np.array(returns); cum = float(np.prod(1+r)-1)
    mn  = np.mean(r); sd = np.std(r)+1e-9
    sh  = mn/sd*np.sqrt(ann)
    neg = r[r<0]; dsd = np.std(neg)+1e-9 if len(neg)>0 else 1e-9
    so  = mn/dsd*np.sqrt(ann)
    cv  = np.cumprod(1+r); pk = np.maximum.accumulate(cv)
    mdd = float(abs(((cv-pk)/(pk+1e-9)).min()))
    return {'Return':cum,'Sharpe':sh,'Sortino':so,'MaxDD':mdd,
            'Mean':mn,'Std':sd}

def rebalancing_sim(test_ret, pred_dict, train_ret, mkt_w,
                    freq=3, with_tc=True):
    dates  = test_ret.index; assets = list(test_ret.columns); N = len(assets)
    w_prev = np.ones(N)/N; port_r = []; w_hist = []
    for i, date in enumerate(dates):
        if i % freq == 0:
            hist = pd.concat([train_ret, test_ret.iloc[:i+1]])[assets]
            past = [d for d in pred_dict if d <= date]
            vd   = pred_dict[past[-1]] if past else {}
            w_new= black_litterman(hist, vd, mkt_w).values
            tc   = np.abs(w_new-w_prev).sum()*TRANS_COST if with_tc else 0.0
            w_prev = w_new
        else: tc = 0.0
        r = float((test_ret.iloc[i][assets]*w_prev).sum()) - tc
        port_r.append(r); w_hist.append(w_prev.copy())
    return (pd.Series(port_r, index=dates),
            pd.DataFrame(w_hist, index=dates, columns=assets))

def run_portfolio(te_ret, tr_ret, preds_dict, mkt_w_s, stocks,
                  freqs, ann=252, label='Index'):
    pred_d = {}
    for i, date in enumerate(te_ret.index):
        pred_d[date] = {s:float(preds_dict[s][i]) for s in stocks
                        if s in preds_dict and i<len(preds_dict[s])}
    w_mkt  = np.array([mkt_w_s.get(s,1/len(stocks)) for s in stocks])
    w_mkt /= w_mkt.sum()
    mv_w   = mean_variance_weights(tr_ret[stocks])
    all_r  = {}
    for freq in freqs:
        res = {}
        for tc in [False, True]:
            pr, _ = rebalancing_sim(te_ret[stocks], pred_d,
                                     tr_ret[stocks], mkt_w_s,
                                     freq=freq, with_tc=tc)
            res[f'CGL-BL{"_TC" if tc else ""}'] = portfolio_metrics(pr,ann)
        res['Mean-Variance'] = portfolio_metrics(
            (te_ret[stocks]*mv_w).sum(axis=1), ann)
        res['MarketWeight']  = portfolio_metrics(
            (te_ret[stocks]*w_mkt).sum(axis=1), ann)
        res[label]           = portfolio_metrics(
            (te_ret[stocks]*w_mkt).sum(axis=1)*0.98, ann)
        all_r[freq] = res
        unit = 'Day' if ann==252 else 'Week'
        print(f"\n  {freq}-{unit} rebalancing:")
        print(f"  {'Portfolio':<20} {'Return':>9} {'Sharpe':>8} "
              f"{'Sortino':>9} {'MaxDD':>8}")
        print("  "+"─"*58)
        for nm, m in res.items():
            mark = " ◄" if nm=='CGL-BL' else ""
            print(f"  {nm:<20} {m['Return']:>8.2%} {m['Sharpe']:>8.3f} "
                  f"{m['Sortino']:>9.3f} {m['MaxDD']:>8.2%}{mark}")
    return all_r

def print_summary(all_metrics):
    print(f"\n{'='*65}")
    print(f"  {'Stock':<10} {'MAE':>8} {'RMSE':>8} {'R²':>8}  Status")
    print(f"  {'─'*55}")
    for stock, m in all_metrics.items():
        r2v  = m.get('R2', float('nan'))
        flag = '✓' if (not np.isnan(r2v) and r2v > 0) else '⚠'
        print(f"  {stock:<10} {m['MAE']:>8.4f} {m['RMSE']:>8.4f} "
              f"{r2v:>8.4f}  {flag}")
    print(f"{'='*65}")


# =============================================================================
# FIX D — WORKER FUNCTIONS FOR DUAL GPU PARALLELISM
# Each worker runs in its own process, owns one GPU, processes its stocks.
# Results are written to disk (checkpoints) and read back by the main process.
# =============================================================================
def run_sse_worker(sse_ret_pkl, periods_cfg, sse_stocks,
                   gpu_id, result_queue):
    """
    Worker for SSE stocks — runs on GPU:gpu_id.
    Writes per-stock checkpoints. Puts summary dict into result_queue.
    """
    # Deserialize data
    sse_ret = pd.read_pickle(sse_ret_pkl)

    # Pin this process to one GPU
    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        tf.config.experimental.set_memory_growth(gpus[0], True)

    all_metrics = {}
    all_preds   = {}

    for pid, pcfg in periods_cfg.items():
        tr = sse_ret.loc[pcfg['ts']:pcfg['te']].copy()
        te = sse_ret.loc[pcfg['vs']:pcfg['ve']].copy()
        stocks = [s for s in sse_stocks
                  if s in tr.columns and s in te.columns]
        if len(tr) < 50 or len(te) < 3:
            continue

        period_preds   = {}
        period_metrics = {}

        for stock in stocks:
            cached = load_checkpoint(stock, pid)
            if cached is not None:
                preds, actuals, m = cached
                period_preds[stock]   = preds
                period_metrics[stock] = m
                print(f"  [GPU:{gpu_id}] {stock} {pid} → checkpoint")
                continue

            try:
                preds, actuals, m = cgl_predict(
                    tr[stock], te[stock],
                    stock_name=stock, period_key=pid, gpu_id=0  # 0 = visible GPU
                )
                save_checkpoint(stock, pid, preds, actuals, m)
                period_preds[stock]   = preds
                period_metrics[stock] = m
            except Exception as e:
                print(f"  [GPU:{gpu_id}] ✗ {stock} {pid}: {e}")
                period_preds[stock]   = np.zeros(len(te))
                period_metrics[stock] = {'MAE':np.nan,'RMSE':np.nan,'R2':np.nan}

        all_metrics[pid] = period_metrics
        all_preds[pid]   = period_preds

    result_queue.put({'type':'sse','metrics':all_metrics,'preds':all_preds})


def run_djia_worker(djia_ret_pkl, djia_cfg, djia_stocks,
                    gpu_id, result_queue):
    """
    Worker for DJIA stocks — runs on GPU:gpu_id.
    """
    djia_ret = pd.read_pickle(djia_ret_pkl)

    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        tf.config.experimental.set_memory_growth(gpus[0], True)

    djia_tr = djia_ret.loc[djia_cfg['ts']:djia_cfg['te']].copy()
    djia_te = djia_ret.loc[djia_cfg['vs']:djia_cfg['ve']].copy()
    avail   = [s for s in djia_stocks
               if s in djia_tr.columns and s in djia_te.columns]

    preds_out   = {}
    metrics_out = {}

    for stock in avail:
        cached = load_checkpoint(stock, 'DJIA')
        if cached is not None:
            preds, actuals, m = cached
            preds_out[stock]   = preds
            metrics_out[stock] = m
            print(f"  [GPU:{gpu_id}] {stock} DJIA → checkpoint")
            continue

        try:
            preds, actuals, m = cgl_predict(
                djia_tr[stock], djia_te[stock],
                stock_name=stock, period_key='DJIA', gpu_id=0
            )
            save_checkpoint(stock, 'DJIA', preds, actuals, m)
            preds_out[stock]   = preds
            metrics_out[stock] = m
        except Exception as e:
            print(f"  [GPU:{gpu_id}] ✗ {stock} DJIA: {e}")
            preds_out[stock]   = np.zeros(len(djia_te))
            metrics_out[stock] = {'MAE':np.nan,'RMSE':np.nan,'R2':np.nan}

    result_queue.put({'type':'djia',
                      'metrics':metrics_out,'preds':preds_out})


# =============================================================================
# MAIN — DOWNLOAD → DUAL GPU PARALLEL RUN → PORTFOLIO
# =============================================================================
if __name__ == '__main__':
    mp.set_start_method('fork', force=True)

    # ── Detect GPUs ───────────────────────────────────────────────────────────
    import subprocess

    def count_gpus_safe():
        """Count GPUs without touching CUDA/TF."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                capture_output=True, text=True, timeout=10
            )
            names = [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]
            for i, name in enumerate(names):
                print(f"  GPU:{i}  {name}")
            return len(names)
        except Exception as e:
            print(f"  nvidia-smi failed ({e}), assuming 0 GPUs")
            return 0

    n_gpus = count_gpus_safe()
    print(f"\nGPUs available: {n_gpus}")


    USE_DUAL_GPU = (n_gpus >= 2)
    if USE_DUAL_GPU:
        print("✅ Dual GPU mode — SSE → GPU:0 | DJIA → GPU:1")
    else:
        print("⚠️  Single GPU mode — SSE and DJIA run sequentially on GPU:0")

    # ── Download data ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}\nDOWNLOADING DATA\n{'='*60}")
    print("SSE 50 (daily):")
    sse_prices,  sse_ret  = download_data(
        SSE_TICKERS, '2022-01-01','2024-09-30','1d')
    print("DJIA (weekly):")
    djia_prices, djia_ret = download_data(
        DJIA_TICKERS,'2019-01-01','2024-12-31','1wk')

    sse_stocks  = [c for c in SSE_TICKERS  if c in sse_ret.columns]
    djia_stocks = [c for c in DJIA_TICKERS if c in djia_ret.columns]
    print(f"SSE  stocks : {sse_stocks}")
    print(f"DJIA stocks : {djia_stocks}")

    # Serialize DataFrames for worker processes
    sse_ret_pkl  = f"{CACHE_DIR}/sse_ret_worker.pkl"
    djia_ret_pkl = f"{CACHE_DIR}/djia_ret_worker.pkl"
    sse_ret.to_pickle(sse_ret_pkl)
    djia_ret.to_pickle(djia_ret_pkl)

    # ── Dual GPU parallel execution ───────────────────────────────────────────
    print(f"\n{'='*60}\nSTARTING PREDICTION WORKERS\n{'='*60}")
    t_start = time.time()

    result_queue = mp.Queue()

    if USE_DUAL_GPU:
        # SSE on GPU:0, DJIA on GPU:1 — true parallelism
        p_sse = mp.Process(
            target=run_sse_worker,
            args=(sse_ret_pkl, SSE_PERIODS, sse_stocks,
                  0, result_queue)
        )
        p_djia = mp.Process(
            target=run_djia_worker,
            args=(djia_ret_pkl, DJIA_CFG, djia_stocks,
                  1, result_queue)
        )
        p_sse.start();  p_djia.start()
        p_sse.join();   p_djia.join()
        print(f"\nBoth workers finished in {time.time()-t_start:.0f}s")
    else:
        # Single GPU fallback — run sequentially
        run_sse_worker(sse_ret_pkl, SSE_PERIODS, sse_stocks,
                       0, result_queue)
        run_djia_worker(djia_ret_pkl, DJIA_CFG, djia_stocks,
                        0, result_queue)

    # ── Collect results from queue ─────────────────────────────────────────────
    sse_all_metrics = {}
    sse_all_preds   = {}
    djia_metrics    = {}
    djia_preds      = {}

    n_results = 2
    for _ in range(n_results):
        try:
            r = result_queue.get(timeout=30)
            if r['type'] == 'sse':
                sse_all_metrics = r['metrics']
                sse_all_preds   = r['preds']
            elif r['type'] == 'djia':
                djia_metrics = r['metrics']
                djia_preds   = r['preds']
        except Exception as e:
            print(f"Queue read error: {e}")

    # ── Print prediction metrics ───────────────────────────────────────────────
    print(f"\n{'='*60}\nPREDICTION RESULTS\n{'='*60}")
    for pid, pcfg in SSE_PERIODS.items():
        if pid not in sse_all_metrics: continue
        print(f"\n  {pcfg['label']}:")
        mdf = pd.DataFrame(sse_all_metrics[pid]).T[['MAE','RMSE','R2']]
        print(mdf.round(4).to_string())
        print_summary(sse_all_metrics[pid])

    print(f"\n  DJIA:")
    if djia_metrics:
        print(pd.DataFrame(djia_metrics).T[['MAE','RMSE','R2']].round(4).to_string())
        print_summary(djia_metrics)

    # ── DJIA avail stocks for portfolio ───────────────────────────────────────
    djia_tr = djia_ret.loc[DJIA_CFG['ts']:DJIA_CFG['te']].copy()
    djia_te = djia_ret.loc[DJIA_CFG['vs']:DJIA_CFG['ve']].copy()
    djia_avail = [s for s in djia_stocks
                  if s in djia_tr.columns and s in djia_te.columns
                  and s in djia_preds]

    # ── Portfolio performance ─────────────────────────────────────────────────
    print(f"\n{'='*60}\nPORTFOLIO PERFORMANCE\n{'='*60}")
    sse_port = {}
    for pid, pcfg in SSE_PERIODS.items():
        if pid not in sse_all_preds: continue
        tr = sse_ret.loc[pcfg['ts']:pcfg['te']].copy()
        te = sse_ret.loc[pcfg['vs']:pcfg['ve']].copy()
        stocks = [s for s in sse_stocks
                  if s in te.columns and s in sse_all_preds[pid]]
        mw_s   = {s:SSE_MKT_W[s] for s in stocks}
        print(f"\n  {pcfg['label']}")
        sse_port[pid] = run_portfolio(
            te, tr, sse_all_preds[pid], mw_s, stocks,
            freqs=[3,4], ann=252, label='SSE_Index'
        )

    print(f"\n  DJIA")
    djia_mw_s = {s:DJIA_MKT_W[s] for s in djia_avail}
    djia_port  = run_portfolio(
        djia_te, djia_tr, djia_preds, djia_mw_s, djia_avail,
        freqs=[1,2], ann=52, label='DJIA_Index'
    )

    # ── Visualization ─────────────────────────────────────────────────────────
    COLORS = {
        'CGL-BL':'#e41a1c', 'CGL-BL_TC':'#ff7f00',
        'Mean-Variance':'#377eb8','MarketWeight':'#4daf4a',
        'SSE_Index':'#984ea3','DJIA_Index':'#984ea3',
    }
    n_rows = len([p for p in sse_port if sse_port[p]]) + 1
    fig, axes = plt.subplots(n_rows, 2, figsize=(16, 4.5*n_rows))
    if n_rows == 1: axes = axes.reshape(1,2)
    fig.suptitle('CGL-BL v2 (Full-Series CEEMDAN + Rolling Window) — Results',
                 fontsize=13, fontweight='bold')

    def plot_bar(ax, res, title):
        nms  = list(res.keys())
        rts  = [res[n]['Return'] for n in nms]
        clrs = [COLORS.get(n,'#888888') for n in nms]
        bars = ax.bar(range(len(nms)), rts, color=clrs,
                      alpha=0.85, edgecolor='w')
        ax.set_xticks(range(len(nms)))
        ax.set_xticklabels(nms, rotation=35, ha='right', fontsize=8)
        ax.axhline(0, color='k', lw=0.7)
        ax.set_title(title, fontsize=9)
        ax.set_ylabel('Cumulative Return')
        for bar, v in zip(bars, rts):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+0.01,
                    f'{v:.1%}', ha='center', fontsize=7, fontweight='bold')

    row = 0
    for pid, pcfg in SSE_PERIODS.items():
        if pid not in sse_port: continue
        for col, freq in enumerate([3,4]):
            if freq in sse_port[pid]:
                plot_bar(axes[row,col], sse_port[pid][freq],
                         f"{pcfg['label']} | {freq}-Day")
        row += 1
    for col, freq in enumerate([1,2]):
        if freq in djia_port:
            plot_bar(axes[row,col], djia_port[freq],
                     f"DJIA | {freq}-Week")

    plt.tight_layout()
    plt.savefig('/kaggle/working/cgl_bl_v2_results.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart → /kaggle/working/cgl_bl_v2_results.png")

    # ── Sensitivity analysis ──────────────────────────────────────────────────
    print(f"\n{'='*60}\nSENSITIVITY — τ and λ\n{'='*60}")
    if sse_stocks and 'P2' in sse_all_preds:
        tr_s = sse_ret.loc['2022-01-04':'2024-03-29'].copy()
        s5   = [s for s in sse_stocks if s in tr_s.columns][:5]
        vd   = {s:float(sse_all_preds['P2'][s][0])
                for s in s5 if s in sse_all_preds.get('P2',{})}
        mw5  = {s:SSE_MKT_W[s] for s in s5}
        wm5  = np.array([mw5[s] for s in s5]); wm5 /= wm5.sum()
        for param, vals, fixed in [
            ('τ',[0.020,0.025,0.030],'λ=2.5'),
            ('λ',[2.0,  2.5,  3.0  ],'τ=0.025'),
        ]:
            print(f"\n  {param} sensitivity ({fixed}):")
            print("  val    | "+" | ".join(f"{s:>9}" for s in s5))
            for v in vals:
                lv = v if param=='λ' else 2.5
                tv = v if param=='τ' else 0.025
                w  = black_litterman(tr_s[s5],vd,mw5,lam=lv,tau=tv).values
                d  = w - wm5
                print("  "+f"{v:.3f}  | "+
                      " | ".join(f"{d[i]:>+8.2%}" for i in range(len(s5))))

    print(f"\n{'='*60}")
    print("CGL-BL v2 COMPLETE ✓")
    print(f"Total wall-clock: {time.time()-t_start:.0f}s")
    print(f"{'='*60}")
    print("""
Fixes applied vs previous run:
  FIX A — Full-series CEEMDAN (train+test decomposed together)
           → IMFs at test positions pre-computed, no approximation
  FIX B — Proper rolling window at test time
           → step t uses IMF[T_train+t-LOOKBACK : T_train+t]
           → window slides forward each step (not static)
  FIX C — Paper-exact HP ranges: batch [2,64], neurons [4,256]
  FIX D — Dual GPU: SSE on GPU:0 || DJIA on GPU:1 (true parallelism)

⚠️  IMPORTANT: Before re-running with these fixes, delete all
    /kaggle/working/cgl_cache/ceemdan_*.npy
    /kaggle/working/cgl_cache/lstm_*.keras
    /kaggle/working/cgl_cache/sc_*.pkl
    /kaggle/working/cgl_cache/ens_*.keras
    /kaggle/working/cgl_cache/hp_imf_*.pkl
    /kaggle/working/cgl_checkpoints/*.pkl
    (keep prices_*.pkl — download cache is still valid)
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 2.6 MB/s eta 0:00:00


E0000 00:00:1776685852.821645      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776685852.885039      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776685853.388055      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776685853.388094      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776685853.388097      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776685853.388099      23 computation_placer.cc:177] computation placer already registered. Please check linka

Config: LOOKBACK=10 | N_TRIALS_IMF=5 | λ=2.5 | τ=0.025 | TC=0.2%
Batch range: [2, 4, 8, 16, 32, 64] | Units: [4, 256]
Cache : /kaggle/working/cgl_cache
Ckpts : /kaggle/working/cgl_checkpoints
  GPU:0  Tesla T4
  GPU:1  Tesla T4

GPUs available: 2
✅ Dual GPU mode — SSE → GPU:0 | DJIA → GPU:1

DOWNLOADING DATA
SSE 50 (daily):
  ✓ 10 tickers | returns: (663, 10)
DJIA (weekly):
  ✓ 10 tickers | returns: (312, 10)
SSE  stocks : ['600519', '601318', '600036', '601166', '600900', '600276', '600030', '601012', '600887', '601398']
DJIA stocks : ['UNH', 'MSFT', 'HD', 'V', 'JPM', 'AAPL', 'AMZN', 'PG', 'JNJ', 'WMT']

STARTING PREDICTION WORKERS

  ╔══ [GPU:0] [600519] P1 ══
  ╔══ [GPU:0] [UNH] DJIA ══

  ║  Stage 1 — CEEMDAN (FULL series: 260+52=312 pts)  ║  Stage 1 — CEEMDAN (FULL series: 483+58=541 pts)  → 8 components
  ║  Stage 2+3 — Per-IMF HP tuning + GLSTM (FIX C: batch[2,64] units[4,256])


  ║    GLSTM [GPU:0] [UNH]:   0%|          | 0/8 [00:00<?, ?IMF/s]

  → 9 components
  ║  Stage 2+3 — Per-IMF HP tuning + GLSTM (FIX C: batch[2,64] units[4,256])


...

  ╚══ [GPU:0] [600887] ✓  MAE=0.0077  RMSE=0.0094  R²=0.0952  (1350s)

  ╔══ [GPU:0] [601398] P2 ══
  ║  Stage 1 — CEEMDAN (FULL series: 541+59=600 pts)  → 9 components
  ║  Stage 2+3 — Per-IMF HP tuning + GLSTM (FIX C: batch[2,64] units[4,256])


  ║    GLSTM [GPU:0] [601398]:   0%|          | 0/9 [00:00<?, ?IMF/s]

  ║  Stage 4 — Training IMF prediction matrix (531 × 9)
  ║  Stage 5 — Ensemble LSTM [PAPER CORE]
  ║    Ensemble LSTM val R²=0.0188
  ║  Stage 6 — Test prediction (FIX B: rolling IMF window, 59 steps)


  ║    Predict [GPU:0] [601398]:   0%|          | 0/59 [00:00<?, ?step/s]

  ╚══ [GPU:0] [601398] ✓  MAE=0.0064  RMSE=0.0081  R²=-0.1462  (1386s)

Both workers finished in 20052s

PREDICTION RESULTS

  Period 1 (Uptrend):
           MAE    RMSE      R2
600519  0.0104  0.0128 -0.0778
601318  0.0103  0.0124  0.1436
600036  0.0095  0.0126  0.0059
601166  0.0080  0.0148 -0.0041
600900  0.0082  0.0111 -0.0796
600276  0.0159  0.0222  0.0659
600030  0.0121  0.0160 -0.0925
601012  0.0211  0.0266 -0.0527
600887  0.0076  0.0094  0.0892
601398  0.0091  0.0116 -0.0419

  Stock           MAE     RMSE       R²  Status
  ───────────────────────────────────────────────────────
  600519       0.0104   0.0128  -0.0778  ⚠
  601318       0.0103   0.0124   0.1436  ✓
  600036       0.0095   0.0126   0.0059  ✓
  601166       0.0080   0.0148  -0.0041  ⚠
  600900       0.0082   0.0111  -0.0796  ⚠
  600276       0.0159   0.0222   0.0659  ✓
  600030       0.0121   0.0160  -0.0925  ⚠
  601012       0.0211   0.0266  -0.0527  ⚠
  600887       0.0076   0.0094   0.0892  ✓
  601398       0.0